# Изменение значения L
Исследуется влияние на изменение значения L (`destinationRange`) для эквализации гистограмм.

По очереди выводятся: нормальное изображение, недоэкспонированное и переэкспонированное (и их гистограммы и CDF). Для каждого выводятся результаты эквализации со значениями L = 256, 192, 128.

In [ ]:

import numpy as np;
import cv2 as cv;
import matplotlib.pyplot as plt;


def histEqualization(img, dest_range):
    h = cv.calcHist(
        [img],      # исходное изображение в виде списка
        [0],        # индекс канала для которого вычисляется гистограмма (0 — в оттенках серого, 1—3 — цвета BGR)
        None,       # маска (None — без маски)
        [256],      # массив размеров гистограммы
        [0, 256]    # диапазон значений яркости
    );
    numPixels = img.shape[0] * img.shape[1];
    
    cdf = makeCdf(h);
    
    # нормализация
    firstNonZero = 0;
    for el in h:
        if el != 0:
            firstNonZero = el;
            break;
    
    sourceRange = numPixels;
    destinationRange = dest_range;
    
    lut = np.zeros_like(h);
    
    for i in range(destinationRange):
        offset = cdf[i] - firstNonZero;
        if offset < 0:
            offset = 0;
        lut[i] = (offset / sourceRange) * destinationRange;
    
    return cv.LUT(img, lut).astype(np.uint8);


def makeCdf(histogram):
    integral = np.zeros_like(histogram);
    integral[0] = histogram[0];
    for i in range(1, integral.shape[0]):
        integral[i] = integral[i - 1] + histogram[i];
    return integral;


images = ['img1.jpg', 'img2.jpg', 'img3.jpg'];
labels = ['Исходное изображение', 'Недоэкспонированное изображение',
          'Переэкспонированное изображение'];
l_vals = [256, 192, 128];

for image_name, label in zip(images, labels):
    img_bgr = cv.imread(image_name);
    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY);
    for l in l_vals:
        equalized_cv = histEqualization(gray, l);

        bins = [i for i in range(256)];

        # Гистограмма исходного изображения
        h_original = cv.calcHist([gray], [0], None, [256], [0, 256]);

        # Гистограмма эквализованного изображения
        h_cv = cv.calcHist([equalized_cv], [0], None, [256], [0, 256]);

        fig, ax = plt.subplots(2, 2, figsize=(12, 8));

        # Исходное изображение
        ax[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255);
        ax[0, 0].set_title('Исходное изображение');
        ax[0, 0].axis('off');

        # Гистограмма и CDF исходного изображения
        ax[0, 1].bar(bins, h_original.flatten());
        cdf_original = makeCdf(h_original).flatten();
        cdf_ax = ax[0, 1].twinx();
        cdf_ax.plot(bins, cdf_original, color='red');
        cdf_ax.set_ylim(0, cdf_original[-1]);
        cdf_ax.set_ylabel("Эмпирическая функция распределения");
        ax[0, 1].set_title('Гистограмма исходного изображения');

        # Эквализованное изображение
        ax[1, 0].imshow(equalized_cv, cmap='gray', vmin=0, vmax=255);
        ax[1, 0].set_title('Эквализованное изображение');
        ax[1, 0].axis('off');

        # Гистограмма и CDF эквализованного изображения
        ax[1, 1].bar(bins, h_cv.flatten());
        cdf_cv = makeCdf(h_cv).flatten();
        cdf_ax = ax[1, 1].twinx();
        cdf_ax.plot(bins, cdf_cv, color='blue');
        cdf_ax.set_ylim(0, cdf_cv[-1]);
        cdf_ax.set_ylabel("Эмпирическая функция распределения");
        ax[1, 1].set_title('Гистограмма после эквализации');

        fig.suptitle(label + ', L = ' + str(l));
        fig.tight_layout();
        plt.show();
